# Particles regroup from spatial location instead of initial position

- Particles are released every 5 days, the grouping is not so obvious. We will now group according to the date when they cross certain threshold.

In [4]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from xhistogram.xarray import histogram as xhist
from pathlib import Path

In [5]:
from dask.distributed import Client
client = Client(n_workers=4, threads_per_worker=3, memory_limit=15e9)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 12,Total memory: 55.88 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36907,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:42137,Total threads: 3
Dashboard: http://127.0.0.1:39953/status,Memory: 13.97 GiB
Nanny: tcp://127.0.0.1:34577,


In [6]:
!echo dask dashboard :D
!echo https://jupyterhub.dkrz.de/user/$USER/levante-spawner-preset/proxy/8787/status

dask dashboard :D
https://jupyterhub.dkrz.de/user/b383184/levante-spawner-preset/proxy/8787/status


In [7]:
stores1 = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/run_2/tracks_v3_11098723").glob("Parcels_run_*_*.zarr"))
stores2  = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/run_2/tracks_v3_45678").glob("Parcels_run_*_*.zarr"))
stores3 = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/run_2/tracks_v3_67891").glob("Parcels_run_*_*.zarr"))
stores4 = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/run_2/tracks_v3_78876").glob("Parcels_run_*_*.zarr"))

stores = stores1+stores2+stores3+stores4
stores[:3]

[PosixPath('/work/bk1450/b383184/Amazon/Mercator/data/run_2/tracks_v3_11098723/Parcels_run_11098723_1993-01-06 00:00:00.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Mercator/data/run_2/tracks_v3_11098723/Parcels_run_11098723_1993-01-11 00:00:00.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Mercator/data/run_2/tracks_v3_11098723/Parcels_run_11098723_1993-01-16 00:00:00.zarr')]

In [2]:
def calc_hist(ds):
    return xhist(
        ds.lon,
        ds.lat,
        bins=[lon_bins, lat_bins],
        dim=["trajectory", ],
        bin_dim_suffix=""
    )

## 1) We concat stores in all time

In [8]:
ds_list = [xr.open_zarr(s) for s in stores]
ds = xr.concat(ds_list,dim='trajectory')

In [9]:
ds = ds.drop_vars(['z','sal','temp','ndry','nrefine'])
ds

<xarray.Dataset> Size: 344GB
Dimensions:     (trajectory: 15485167, obs: 925)
Coordinates:
  * obs         (obs) int32 4kB 0 1 2 3 4 5 6 7 ... 918 919 920 921 922 923 924
  * trajectory  (trajectory) int64 124MB 0 3 4 6 8 ... 16394 16395 16396 16399
Data variables:
    lat         (trajectory, obs) float64 115GB dask.array<chunksize=(10061, 185), meta=np.ndarray>
    lon         (trajectory, obs) float64 115GB dask.array<chunksize=(10061, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 115GB dask.array<chunksize=(10061, 185), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        SampleParticleDeleteNorthSampleTSAdvectionRK4_3D_...
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [10]:
num_valid_obs_steps = int(ds.lat.notnull().any('trajectory').sum().compute().data[()])
ds = ds.isel(obs=slice(None, num_valid_obs_steps))
ds

<xarray.Dataset> Size: 276GB
Dimensions:     (trajectory: 15485167, obs: 741)
Coordinates:
  * obs         (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 734 735 736 737 738 739 740
  * trajectory  (trajectory) int64 124MB 0 3 4 6 8 ... 16394 16395 16396 16399
Data variables:
    lat         (trajectory, obs) float64 92GB dask.array<chunksize=(10061, 185), meta=np.ndarray>
    lon         (trajectory, obs) float64 92GB dask.array<chunksize=(10061, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 92GB dask.array<chunksize=(10061, 185), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        SampleParticleDeleteNorthSampleTSAdvectionRK4_3D_...
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [11]:
ds = ds.assign(start_time=ds.time.isel(obs=0).compute())
ds

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 9.79 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


<xarray.Dataset> Size: 276GB
Dimensions:     (trajectory: 15485167, obs: 741)
Coordinates:
  * obs         (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 734 735 736 737 738 739 740
  * trajectory  (trajectory) int64 124MB 0 3 4 6 8 ... 16394 16395 16396 16399
Data variables:
    lat         (trajectory, obs) float64 92GB dask.array<chunksize=(10061, 185), meta=np.ndarray>
    lon         (trajectory, obs) float64 92GB dask.array<chunksize=(10061, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 92GB dask.array<chunksize=(10061, 185), meta=np.ndarray>
    start_time  (trajectory) datetime64[ns] 124MB 1993-01-06 ... 2013-12-28
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        SampleParticleDeleteNorthSampleTSAdvectionRK4_3D_...
    parcels_mesh:           spherical
    parcels_version:        3.1.2

## 2) Find when they crossed 2.5N,4N,5N,..12N
- Group according the date when they crossed the threshold
- start_time should be then time_treshold

In [12]:
thr_list = [2.5, 3.5, 5, 6, 7, 8, 9, 10, 11, 12]

In [13]:
def threshold_group(thr_list,ds):
    ds_m = ds
    hist_thr = {} 
    
    for i in range(len(thr_list)):
        cross = (ds.lat.shift(obs=1) < thr_list[i]) & (ds.lat >= thr_list[i])
        crossed = cross.any('obs') # false if it never reaches the threshold
        idx = cross.argmax('obs') # obs index of first crossing
        # ds = ds.assign(**{f'start_{thr_list[i]}N': ds.time.isel(obs=idx)})
        #particles that never cross threshold are NaN
        ds = ds.assign(**{f'start_{thr_list[i]}N': ds.time.isel(obs=idx).where(crossed)}) 
        #Particles that don't cross go to month 13
        ds_m = ds_m.assign(**{f'month_start_{thr_list[i]}N': ds[f'start_{thr_list[i]}N'].dt.month.fillna(13)})

        #histogram
        ds_m = ds_m.assign(thr_month=ds_m[f'month_start_{thr_list[i]}N'])

        hist_temporal = ds_m.groupby(["thr_month"]).apply(calc_hist)
        hist_temporal = hist_temporal.isel(obs=slice(1, None)).coarsen(obs=4 * 5, boundary="trim").sum()
        
        # store the hist
        hist_thr[f'hist_{thr_list[i]}N'] = hist_temporal

    
    return ds, ds_m, hist_thr

# ds_m = ds.assign(start_month=ds.start_time.dt.month)
# ds_m = ds_m.assign(thr_month=ds.time_treshold.dt.month)
# ds_m

In [14]:
lat_min=ds.lat.min().compute().data[()]
lon_min=ds.lon.min().compute().data[()]

lat_max=ds.lat.max().compute().data[()]
lon_max=ds.lon.max().compute().data[()]

lat_bins = np.linspace(lat_min, lat_max, 30)
lon_bins = np.linspace(lon_min, lon_max, 80)

In [15]:
nds,nds_m,hist_thr = threshold_group(thr_list,ds)
hist_thr

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 11.79 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


IndexError: vindex does not support indexing with dask objects. Call compute on the indexer first to get an evalurated array. Got: (array([       0,        1,        2, ..., 15485164, 15485165, 15485166],
      shape=(15485167,)), dask.array<arg_agg-aggregate, shape=(15485167,), dtype=int64, chunksize=(10116,), chunktype=numpy.ndarray>)

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cf
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.patches as mpatches

In [ ]:
hist_thr[f'hist_{thr_list[0]}N']

In [ ]:
for ii in range(len(thr_list)):
    
    hist_ = hist_thr[f'hist_{thr_list[ii]}N']
    
    proj = ccrs.PlateCarree()
    
    fig, ax = plt.subplots(
        5, 3,
        figsize=(10, 8),                      # a bit wider, not so tall
        subplot_kw=dict(projection=proj)
    )
    ax = ax.ravel()
    
    for i in range(12+1):
        dd = hist_.sel(thr_month=i+1).sum("obs")
    
        # plot
        pcm = np.log10(dd).plot.contourf(
            x='lon', y='lat', ax=ax[i], transform=proj,
            add_colorbar=False, cmap='Spectral_r'
        )
    
        ax[i].coastlines()
        ax[i].add_feature(cf.LAND, facecolor='#8e9497ff', zorder=0)
        ax[i].set_title(i+1, fontsize=10)
    
        # ticks & labels
        ax[i].set_xticks(
            np.arange(np.floor(dd.lon.min()), np.ceil(dd.lon.max()) + 1e-6, 10),
            crs=proj
        )
        ax[i].set_yticks(
            np.arange(np.floor(dd.lat.min()), np.ceil(dd.lat.max()) + 1e-6, 5),
            crs=proj
        )
        ax[i].xaxis.set_major_formatter(
            LongitudeFormatter(number_format='.0f', degree_symbol='°')
        )
        ax[i].yaxis.set_major_formatter(
            LatitudeFormatter(number_format='.0f', degree_symbol='°')
        )
        ax[i].tick_params(labelsize=8)
        ax[i].set_xlabel('')
        ax[i].set_ylabel('')
    
    # Tighter spacing: reduce both wspace and hspace
    fig.subplots_adjust(
        left=0.05, right=0.95,
        bottom=0.08, top=0.90,
        wspace=0.15, hspace=0.50
    )
    
    # Optional: one shared colorbar at bottom
    cbar = fig.colorbar(
        pcm, ax=ax, orientation='horizontal',
        fraction=0.05, pad=0.10
    )
    cbar.ax.set_xlabel('log10(counts)', fontsize=10)
    
    fig.suptitle(f'Threshold: {thr_list[ii]}°N', fontsize=14, fontweight='bold', y=0.95)
    
    plt.show()

In [ ]:
for ii in range(len(thr_list)):
    
    hist_ = hist_thr[f'hist_{thr_list[ii]}N']
    
    proj = ccrs.PlateCarree()
    
    fig, ax = plt.subplots(
        4, 3,
        figsize=(10, 8),                      # a bit wider, not so tall
        subplot_kw=dict(projection=proj)
    )
    ax = ax.ravel()
    
    for i in range(12+1):
        dd = hist_.sel(thr_month=i+1).sum("obs")
    
        # plot
        pcm = np.log10(dd).plot(
            x='lon', y='lat', ax=ax[i], transform=proj,
            add_colorbar=False, cmap='Spectral_r'
        )
    
        ax[i].coastlines()
        ax[i].add_feature(cf.LAND, facecolor='#8e9497ff', zorder=0)
        ax[i].set_title(i+1, fontsize=10)
    
        # ticks & labels
        ax[i].set_xticks(
            np.arange(np.floor(dd.lon.min()), np.ceil(dd.lon.max()) + 1e-6, 10),
            crs=proj
        )
        ax[i].set_yticks(
            np.arange(np.floor(dd.lat.min()), np.ceil(dd.lat.max()) + 1e-6, 5),
            crs=proj
        )
        ax[i].xaxis.set_major_formatter(
            LongitudeFormatter(number_format='.0f', degree_symbol='°')
        )
        ax[i].yaxis.set_major_formatter(
            LatitudeFormatter(number_format='.0f', degree_symbol='°')
        )
        ax[i].tick_params(labelsize=8)
        ax[i].set_xlabel('')
        ax[i].set_ylabel('')
    
    # Tighter spacing: reduce both wspace and hspace
    fig.subplots_adjust(
        left=0.05, right=0.95,
        bottom=0.08, top=0.90,
        wspace=0.15, hspace=0.50
    )
    
    # Optional: one shared colorbar at bottom
    cbar = fig.colorbar(
        pcm, ax=ax, orientation='horizontal',
        fraction=0.05, pad=0.10
    )
    cbar.ax.set_xlabel('log10(counts)', fontsize=10)
    
    fig.suptitle(f'Threshold: {thr_list[ii]}°N', fontsize=14, fontweight='bold', y=0.95)
    
    plt.show()